In this script we show how to use regression LFM for portfolio return stress-testing in a 2-dimensional stock market.

In [1]:
from numpy import array, average, cov, diag, sqrt, sum
from statsmodels.multivariate.factor import Factor
from pandas import read_csv, to_datetime

In [3]:
# find the ROOT & DATA folder.

from pathlib import Path
import sys

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "requirements.txt").exists())
DATA_DIR = ROOT / "data"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))  # lets 'from arpym...' work from any subfolder

Load Data

From the database db_stocks_SP500/SPX we upload the S&P 500 index values and compute daily returns.

In [5]:
# Loading the SPX.csv file into a pandas DataFrame, declaring 'date' as a column of datetimes that could be of mixed format
db_spx = read_csv(DATA_DIR /"SPX.csv", parse_dates=['date'], date_format='mixed', index_col=0)

# Filter the SPX prices into a 2-week window (between 16th March -30th March 2012) and flatten into a 1D array or prices
# extract the values with the dates that we want to look at
v_spx = db_spx[(db_spx.index >= to_datetime('16-03-2012', dayfirst=True)) &
               (db_spx.index <= to_datetime('30-03-2012', dayfirst=True))].values.reshape(-1)

# Our method for computing the daily linear returns of the S&P 500: (price today / price yesterday) - 1
# The value today / value yesterday - 1
z_t = v_spx[1:]/v_spx[:-1] - 1
# round each return off to 7 decimal places
# what we obtain is a Time Series of 10 Linear Returns and that is our 'Factor' variable
print('z_t =', z_t.round(7))


z_t = [ 0.0039739 -0.0030005 -0.0018712 -0.0072066  0.0031089  0.0138858
 -0.0028168 -0.0049415 -0.0016079  0.0036985]


From the database db_stocks_sp we upload AON and CVC stock values, and compute portfolio returns
The target, built from AON + CVC weighted

In [5]:

# We do something very similar for portfolio daily returns , we extract the data for the same dates and for each of the
# stocks we do the same computation - we get the linear returns for each of the stocks  and then we compute the portfolio
# returns using the weights that we specify

# telling pandas to only use these 3 (out of 430+) columns from this csv file
# use the columns names from row 1
# Load AON and CVC price columns from row 2 onward (row 1 is a title row), using date as index
db_stocks_sp = read_csv("data/db_stocks_sp.csv", usecols=['date', 'AON', 'CVC'],
                     parse_dates=True, index_col=0, header=1, dayfirst=True, date_format='mixed')

# stock order used throughout this notebook: [CVC, AON] -,ust match w_tnow below
stock_order = ['CVC', 'AON']

# Filter to 16-30 Mar 2012 window; select columns by name in stock_order (avoids positional index bugs)
v_stocks = db_stocks_sp.loc[(db_stocks_sp.index >= to_datetime('16-03-2012', dayfirst=True)) &
                        (db_stocks_sp.index <= to_datetime('30-03-2012', dayfirst=True)), stock_order].values

# The returns of 2 stocks
# Daily linear returns columns match stock_order [CVC, AON]: (price today / price yesterday) - 1
# The computations for the returns of the 2 stocks
r_n_t = v_stocks[1:, :]/v_stocks[:-1, :] - 1

# Current portfolio weights
# In same order as stock_order: [CVC, AON]
# we can change this if we want to play around with the code and see what happens
w_tnow = array([0.42268011, 0.57731989])

# Portfolio return each day = weighted sum of returns, using stock_order (eq. 25.6)
# Then to obtain the portfolio returns we use numpy's sum() function
# we weight each the returns and then add them all together
x_t = sum(w_tnow * r_n_t, axis=1)

# From this we get our Time Series o portfolio returns
# Display portfolio return series, rounded for readability
print('x_t =', x_t.round(7))

x_t = [ 0.0029476 -0.0066345  0.0015802 -0.0029019  0.0028408  0.0149615
 -0.0095231  0.0031261 -0.0017765 -0.0023666]


Compute Original Regression Loading and Shift
Fitting the regression line and evaluate it at a Hypothetical point z = -0.3%
To get a predicted portfolio return

In [13]:
# The formula for the optimal regression loading and shift
# These depend only on the joint mean and the joint covariance

# Sample mean of [portfolio return, S&P return] — estimate of mu*_{X,Z} (eq. 25.11)
# we combine the factor(z_t) and target(x_t) realizations, into a numpy array, then we use the average function
# over the appropriate axis.
mu_xz = average(array([x_t, z_t]), axis=1)

print(mu_xz)
# The first Mean is the Mean of our target, the 2nd mean is the mean of our factor - both sample means
# - [0.00022537 0.00032225]

# covariance matrix on [portfolio return, S&P return]
# estimating the covariance matrix
sigma2_xz = cov(array([x_t, z_t]))
print()
print(sigma2_xz)
print()
# This produces a 2 x 2 matrix
# [[4.49167498e-05 3.01016382e-05]
# [3.01016382e-05 3.66988976e-05]]
# the variance of the target in the top left,
# the variance of the factor, bottom right and
# the cross covariance in the of diagonals
# The cross covariance is the most interesting part of this that we have created so far,
# Its one of the major factors in driving our loading, or .... Beta.
# The formula is this cross covariance (3.01016382e-05) normalised by the variance of the factor (3.66988976e-05)
# so how much cross covariance per unit of variance of the unit factor in the 1 dimensional case.

print()
print("Our Optimal coefficients: -----")
# factor loading - This is the slope of the regression line
# OLS slope: beta_hat = Cov(x,z) / Var(z) — factor loading (eq. 25.9)
# sigma2_xz[0,1] — the covariance between portfolio returns and S&P returns, how much they move together.
# sigma2_xz[1,1] — the variance of S&P returns alone, how much market moves on its own.
# dividing first by second - calculates how much they move together per unit of the markets own movement.
beta_reg = sigma2_xz[0,1] / sigma2_xz[1,1]
print('beta_reg =', abs(beta_reg).round(7))

# shift - The Intercept - computed using the means
# the shift terms that are chosen so that the shift has a zero mean so all of the shift
# is within the alpha term
# it is the exposure * the mean of our factors mu_xz[1], subtracted from the mean of our target mu_xz[0]
# mu_xz[0] — mean of x_t (portfolio return).
# mu_xz[1] — mean of z_t (S&P return).
alpha_reg = mu_xz[0] - beta_reg * mu_xz[1]
print('alpha_reg =', abs(alpha_reg).round(7))
print()

# Stress test: predict portfolio return if S&P falls 0.3% (eq. 25.10)
# point statement
# This is hypothetical scenario: "suppose the S&P drops 0.3% tomorrow" (note: −0.003 as a decimal = −0.3%).
# Inference
z = -0.003

# Stress test point prediction when considering 3% drop
# x_bar_point = alpha_reg + beta_reg*z
# plugs that hypothetical z into your fitted regression line (alpha_reg + beta_reg * z)
# to predict what the portfolio return would be
# under that scenario. This is exactly equation 25.10's stress test: α̂ + β̂·z.
# The actual regression line...

# fitted shift = alpha_reg
# exposures beta_reg
# times our "what if value" for the factor
x_bar_point = alpha_reg + beta_reg * z
print("If the S&P 500 goes down by 0.3% our portfolio reaction will be: ")
print('x_bar_point =', x_bar_point.round(7))


[0.00022537 0.00032225]

[[4.49167498e-05 3.01016382e-05]
 [3.01016382e-05 3.66988976e-05]]


Our Optimal coefficients: -----
beta_reg = 0.8202328
alpha_reg = 3.9e-05

If the S&P 500 goes down by 0.3% our portfolio reaction will be: 
x_bar_point = -0.0024997


Unearth hidden factors via regression factor scoring (Chapter 26.d Factor Analysis)
Is there a hidden factor, unobserved factor driving both the S&P and my portfolio, that I could discover
purely from how correlated they are - without telling the model 'The S&P is the factor'


In [14]:
# Building out the target correlation matrix that will be passed to the Factor Analysis model. (next step)
# As the Factor Anlaysis model requires rescaled values for fitting.
# Standardize covariance matrix to correlation matrix: corr = D^-1 Σ D^-1, D = diag(std devs)
# sigma2_xz = 2x2 covariance matrix
# diag() pulls out the diagonal entries from the 2x2 matrix - top left to btm right
# sqrt() takes the square root of each of the 2 variances, giving standard deviations instead of variances
# 1/sqrt() inverts the 2 numbers -> 1/std(x), 1/std(z)
# diag() takes the 1D array and builds a brand new 2D matrix, putting values on diagonals everywhere else 0's
# @sigma2_xz@ sandwiching 2 matrix multiplications with our covariance matrix in between
# giving us a correlation matrix diagonals are all 1's (top left -> btm right) a variable always correlates perfectly
# with itself, other diagonal becomes the correlation coefficient between X and Z (between -1 and 1)
cr_xz = (diag(1/sqrt(diag(sigma2_xz))))@sigma2_xz@(diag(1/sqrt(diag(sigma2_xz))))

# MLF fitted model
# Fit ML "Factor Analysis" model with 2 latent factors from the correlation matrix
# Factor Analysis operates on a correlation matrix and not raw values as larger numbers will dominate the fitted factors
# returns a fitted Factor Analysis model.
# hardcoded n_factors = 2, assuming the model will fit perfectly
# mlf2 is actually what was give in the lab notebook
# but created mlf1 helps identify that we are being too flexible with the model by allowing 2 factors

# AI explanation
# "I ran the factor analysis example with n_factor=2 on the 2-variable (X, Z) case from the lab, and then compared it
# against n_factor=1 to sanity-check it. With 2 factors, the second factor's loadings came out near-zero
# (~-0.003 and 0.002), while the first factor's loadings were almost identical to the entire 1-factor model's
# solution (~-0.86 for both).
# The uniqueness values were also nearly the same between the two fits (~0.26 either way).
# My interpretation is that with only 2 observed variables, requesting 2 factors leaves the model with no spare
# degrees of freedom to actually test the factor structure — it's a fully saturated case, so the second factor isn't
# adding genuine explanatory power, it's just filling the extra flexibility the model was given.
# Is that the right way to think about it, or is there a reason the lab example uses n_factor=2 that I'm missing?"
mlf1 = Factor(n_factor=1, corr=cr_xz, method='ml').fit()
mlf2 = Factor(n_factor=2, corr=cr_xz, method='ml').fit()

print()
print()
print('1-factor loadings:\n', mlf1.loadings)
print('1-factor uniqueness:\n', mlf1.uniqueness)
print()
print('2-factor loadings:\n', mlf2.loadings)
print('2-factor uniqueness:\n', mlf2.uniqueness)
print()
print()

# Re-Scaled Loadings
# Rescale standardized loadings back to original units: beta = D @ loadings
# diag(sigma2_xz) pulls the variance diagonal out of the 2x2 matrix
# sqrt(...) converts the variances into standard deviations [std(x), std(y)]
# diag() takes the 1D array and converts it back into a 2D diag matrix
# @mlf2.loadings matrix multiplies the std by the fitted loadings, bringing back to real world units again
# un-standardizing the values
beta = diag(sqrt(diag(sigma2_xz)))@mlf2.loadings

# rescaled variance
# diag(sigma2_xz) pulls the diagonal from the 2D cov matrix
#  mlf2.uniqueness this is the fitted models estimate of what proportion of each variables variance is not explained by
# common factors
# element wise multiplication
delta = diag(sigma2_xz) * mlf2.uniqueness
print()
print('Delta shape =', delta.shape)
print()
print('Beta shape =', beta.shape)
print()
print('sigma2_xz shape =', sigma2_xz.shape)
print()
print('(z_t.reshape(-1,1) - mu_xz shape', (z_t.reshape(-1,1) - mu_xz).shape)
print()

# factor scores
# Estimate factor score time series via regression scoring formula (using centered data)
# z_t.reshape(-1,1) takes the 1D array of S&p returns and reshapes it into column vector shape (10,1) and not (10,)
# - mu_xz subtracts the mean vector from z_t.   this centers the data instead of raw returns.
# .T transposes the result flipping the rows and columns turning from shape (1,2) into (2,10)
# beta / sigma2_xz elementwise division of the rescaled loadings
# @ matrix multiplication so (2,2) @ (10,2) gives me (2,10) 2 rows (one per factor) by 10 columns (one per day)
# uses the fitted model to work backwards from what was observed to estimate what the hidden facto's value
# must have been on  each day

# There is a potential shape issue here...
# z_t is a (10,) array 9just the S&P returns) and after reshape it becomes (10,1).
# mu_xz is a (2,) array the means of X-t and z_t, subtracting a (10,1) array triggers Numpy broadcasting rather than
# raising an error, producing a (10,2) result
# but column[0] of that result is z_t - mean(x,t), mixing the S&P return series with the Portfolios mean.
# This doesnt look like it should be meaningful quantity for factor scoring.
# My understanding is this line should probably center z_t against its own mean only, something like
# z_t.reshape(-1,1) - mu_xz[1]
# though i want to check whether I'm missing the intent behind the current version before assuming its an error.
h_t_fa = beta/sigma2_xz@(z_t.reshape(-1,1) - mu_xz).T
# Display the second latent factor's estimated score time series
print('h_t_fa \n', h_t_fa[1, :].round(7))



1-factor loadings:
 [[0.86117438]
 [0.86093102]]
1-factor uniqueness:
 [0.25837868 0.25879778]

2-factor loadings:
 [[-8.61103432e-01 -4.47570692e-04]
 [-8.61002193e-01  4.47925218e-04]]
2-factor uniqueness:
 [0.25850068 0.25867502]



Delta shape = (2,)

Beta shape = (2, 2)

sigma2_xz shape = (2, 2)

(z_t.reshape(-1,1) - mu_xz shape (10, 2)

h_t_fa 
 [-0.6492604  0.558728   0.3631232  1.287223  -0.499442  -2.3660357
  0.5269019  0.8949133  0.317524  -0.6015603]


# Conclusion......

Still to write up
